# Provider Daily Economics

This notebook uses the research mart to analyze daily provider token volume and estimated revenue.

Revenue here is an estimate based on observed token activity plus as-of OpenRouter catalog pricing snapshots. It is not audited provider revenue.


In [20]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

BASE_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
SRC_DIR = BASE_DIR / "src"
for path in (BASE_DIR, SRC_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print(f"Base dir: {BASE_DIR}")


Base dir: /Users/henrywzh/Desktop/Quant/alternative-data


In [21]:
from research_data.api import build_daily_provider_economics, provider_revenue_daily, provider_tokens_daily

provider_economics = build_daily_provider_economics(base_dir=BASE_DIR, refresh=False)
print("Rows:", len(provider_economics))
provider_economics.head()


Rows: 8863


,usage_date,provider_slug,provider_name,model_permaslug,total_tokens,prompt_tokens,completion_tokens,estimated_revenue,pricing_snapshot_ts,pricing_prompt,pricing_completion,pricing_join_status
0,2026-01-16,anthropic,Anthropic,anthropic/claude-3-5-haiku,1.413208e+09,0.0,0.0,1234.578847,2026-04-19T13:39:19Z,8.000000e-07,0.000004,matched_model_median
1,2026-01-16,anthropic,Anthropic,anthropic/claude-3-7-sonnet-20250219,4.092200e+09,0.0,0.0,13406.048533,2026-04-19T13:39:19Z,3.000000e-06,0.000015,matched_model_median
2,2026-01-16,anthropic,Anthropic,anthropic/claude-3-7-sonnet-20250219:thinking,2.597868e+08,0.0,0.0,851.061645,2026-04-19T13:39:19Z,3.000000e-06,0.000015,matched_model_median
3,2026-01-16,anthropic,Anthropic,anthropic/claude-3-haiku,2.794971e+08,0.0,0.0,76.302701,2026-04-19T13:39:19Z,2.500000e-07,0.000001,matched_model_median
4,2026-01-16,anthropic,Anthropic,anthropic/claude-3.5-sonnet,7.405579e+08,0.0,0.0,2426.067762,2026-04-19T13:39:19Z,3.000000e-06,0.000015,fallback_provider_median


## Selected Providers


In [22]:
selected_providers = ["openai", "anthropic", "google"]
tokens_daily = provider_tokens_daily(selected_providers, base_dir=BASE_DIR, refresh=False)
revenue_daily = provider_revenue_daily(selected_providers, base_dir=BASE_DIR, refresh=False)

print(tokens_daily.tail(10).to_string(index=False))
print(revenue_daily.tail(10).to_string(index=False))


usage_date provider_slug provider_name                model_permaslug  total_tokens  prompt_tokens  completion_tokens  estimated_revenue  pricing_snapshot_ts  pricing_prompt  pricing_completion  pricing_join_status
2026-04-19        google        Google google/gemma-4-31b-it-20260402  2.552908e+09            0.0                0.0         346.557317 2026-04-19T13:39:19Z    1.300000e-07        3.800000e-07 matched_model_median
2026-04-19        openai        OpenAI openai/gpt-4.1-mini-2025-04-14  1.417312e+09            0.0                0.0         606.042420 2026-04-19T13:39:19Z    4.000000e-07        1.600000e-06 matched_model_median
2026-04-19        openai        OpenAI             openai/gpt-4o-mini  3.613132e+09            0.0                0.0         579.365692 2026-04-19T13:39:19Z    1.500000e-07        6.000000e-07 matched_model_median
2026-04-19        openai        OpenAI   openai/gpt-5-mini-2025-08-07  1.558728e+09            0.0                0.0         452.420830 202

In [23]:
provider_summary = (
    revenue_daily.groupby("provider_slug", as_index=False)
    .agg(
        total_tokens=("total_tokens", "sum"),
        estimated_revenue=("estimated_revenue", "sum"),
        priced_rows=("pricing_join_status", lambda s: int((s.str.startswith("matched")).sum())),
        total_rows=("pricing_join_status", "count"),
    )
    .sort_values("estimated_revenue", ascending=False)
)
provider_summary


,provider_slug,total_tokens,estimated_revenue,priced_rows,total_rows
0,anthropic,2.984658e+13,1.164276e+08,772,847
1,google,3.426112e+13,1.821885e+07,789,847
2,openai,1.924702e+13,1.472703e+07,848,848


In [24]:
daily_rollup = (
    revenue_daily.groupby(["usage_date", "provider_slug"], as_index=False)
    .agg(
        total_tokens=("total_tokens", "sum"),
        estimated_revenue=("estimated_revenue", lambda s: s.sum(min_count=1)),
        priced_tokens=("total_tokens", lambda s: s[revenue_daily.loc[s.index, "estimated_revenue"].notna()].sum()),
    )
)
daily_rollup["priced_token_share"] = daily_rollup["priced_tokens"] / daily_rollup["total_tokens"]

daily_rollup


,usage_date,provider_slug,total_tokens,estimated_revenue,priced_tokens,priced_token_share
0,2026-01-16,anthropic,1.784128e+11,688261.287258,1.784128e+11,1.0
1,2026-01-16,google,2.952748e+11,146369.910994,2.952748e+11,1.0
2,2026-01-16,openai,1.210228e+11,60386.595647,1.210228e+11,1.0
3,2026-01-17,anthropic,1.178014e+11,455645.691040,1.178014e+11,1.0
4,2026-01-17,google,2.845527e+11,198471.521567,2.845527e+11,1.0
...,...,...,...,...,...,...
277,2026-04-18,google,4.053390e+11,208506.916175,4.053390e+11,1.0
278,2026-04-18,openai,2.555712e+11,371306.979714,2.555712e+11,1.0
279,2026-04-19,anthropic,6.474324e+10,267054.219007,6.474324e+10,1.0
280,2026-04-19,google,6.168421e+10,32665.772493,6.168421e+10,1.0


In [33]:
revenue_daily[revenue_daily.provider_slug == "google"]['pricing_join_status'].value_counts()

pricing_join_status
matched_model_median        789
fallback_provider_median     58
Name: count, dtype: int64

In [79]:
from research_data.marts import load_dataset
raw_openrouter_models = load_dataset("raw_openrouter_models", base_dir=BASE_DIR).dropna(axis=1, how='all')
raw_openrouter_models['created_at'] = pd.to_datetime(raw_openrouter_models['created_at'], unit='s')
raw_openrouter_models

/var/folders/y9/2slpdtcd62sccy809rd8q1pc0000gn/T/ipykernel_9031/1974555131.py:3: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  raw_openrouter_models['created_at'] = pd.to_datetime(raw_openrouter_models['created_at'], unit='s')


,dataset_id,source_url,source_run_id,scraped_at,created_at,model_id,snapshot_ts,model_name,context_length,architecture,pricing_prompt,pricing_completion,top_provider_id
0,raw_openrouter_models,https://openrouter.ai/api/v1/models,20260415T225352Z-85a00c49,2026-04-15T22:53:52.430405Z,2025-08-08 16:03:40,ai21/jamba-large-1.7,2026-04-15T22:53:52.430405Z,AI21: Jamba Large 1.7,256000.0,text->text,2.000000e-06,8.000000e-06,256000.0
1,raw_openrouter_models,https://openrouter.ai/api/v1/models,20260415T225352Z-85a00c49,2026-04-15T22:53:52.430405Z,2025-02-04 19:32:37,aion-labs/aion-1.0,2026-04-15T22:53:52.430405Z,AionLabs: Aion-1.0,131072.0,text->text,4.000000e-06,8.000000e-06,131072.0
2,raw_openrouter_models,https://openrouter.ai/api/v1/models,20260415T225352Z-85a00c49,2026-04-15T22:53:52.430405Z,2025-02-04 19:25:07,aion-labs/aion-1.0-mini,2026-04-15T22:53:52.430405Z,AionLabs: Aion-1.0-Mini,131072.0,text->text,7.000000e-07,1.400000e-06,131072.0
3,raw_openrouter_models,https://openrouter.ai/api/v1/models,20260415T225352Z-85a00c49,2026-04-15T22:53:52.430405Z,2026-02-23 21:15:06,aion-labs/aion-2.0,2026-04-15T22:53:52.430405Z,AionLabs: Aion-2.0,131072.0,text->text,8.000000e-07,1.600000e-06,131072.0
4,raw_openrouter_models,https://openrouter.ai/api/v1/models,20260415T225352Z-85a00c49,2026-04-15T22:53:52.430405Z,2025-02-04 19:18:38,aion-labs/aion-rp-llama-3.1-8b,2026-04-15T22:53:52.430405Z,AionLabs: Aion-RP 1.0 (8B),32768.0,text->text,8.000000e-07,1.600000e-06,32768.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
695,raw_openrouter_models,https://openrouter.ai/api/v1/models,20260419T133919Z-01363e45,2026-04-19T13:39:19.243669Z,2026-01-19 14:45:13,z-ai/glm-4.7-flash,2026-04-19T13:39:19.243669Z,Z.ai: GLM 4.7 Flash,202752.0,text->text,6.000000e-08,4.000000e-07,<NA>
696,raw_openrouter_models,https://openrouter.ai/api/v1/models,20260419T133919Z-01363e45,2026-04-19T13:39:19.243669Z,2026-02-11 16:59:42,z-ai/glm-5,2026-04-19T13:39:19.243669Z,Z.ai: GLM 5,80000.0,text->text,7.200000e-07,2.300000e-06,<NA>
697,raw_openrouter_models,https://openrouter.ai/api/v1/models,20260419T133919Z-01363e45,2026-04-19T13:39:19.243669Z,2026-03-15 14:06:13,z-ai/glm-5-turbo,2026-04-19T13:39:19.243669Z,Z.ai: GLM 5 Turbo,202752.0,text->text,1.200000e-06,4.000000e-06,<NA>
698,raw_openrouter_models,https://openrouter.ai/api/v1/models,20260419T133919Z-01363e45,2026-04-19T13:39:19.243669Z,2026-04-07 16:07:05,z-ai/glm-5.1,2026-04-19T13:39:19.243669Z,Z.ai: GLM 5.1,202752.0,text->text,9.500000e-07,3.150000e-06,<NA>


In [80]:
llm_models_pricing = raw_openrouter_models[['created_at', 'snapshot_ts', 'model_id', 'model_name', 'context_length', 'pricing_prompt', 'pricing_completion']].copy()
llm_models_pricing

,created_at,snapshot_ts,model_id,model_name,context_length,pricing_prompt,pricing_completion
0,2025-08-08 16:03:40,2026-04-15T22:53:52.430405Z,ai21/jamba-large-1.7,AI21: Jamba Large 1.7,256000.0,2.000000e-06,8.000000e-06
1,2025-02-04 19:32:37,2026-04-15T22:53:52.430405Z,aion-labs/aion-1.0,AionLabs: Aion-1.0,131072.0,4.000000e-06,8.000000e-06
2,2025-02-04 19:25:07,2026-04-15T22:53:52.430405Z,aion-labs/aion-1.0-mini,AionLabs: Aion-1.0-Mini,131072.0,7.000000e-07,1.400000e-06
3,2026-02-23 21:15:06,2026-04-15T22:53:52.430405Z,aion-labs/aion-2.0,AionLabs: Aion-2.0,131072.0,8.000000e-07,1.600000e-06
4,2025-02-04 19:18:38,2026-04-15T22:53:52.430405Z,aion-labs/aion-rp-llama-3.1-8b,AionLabs: Aion-RP 1.0 (8B),32768.0,8.000000e-07,1.600000e-06
...,...,...,...,...,...,...,...
695,2026-01-19 14:45:13,2026-04-19T13:39:19.243669Z,z-ai/glm-4.7-flash,Z.ai: GLM 4.7 Flash,202752.0,6.000000e-08,4.000000e-07
696,2026-02-11 16:59:42,2026-04-19T13:39:19.243669Z,z-ai/glm-5,Z.ai: GLM 5,80000.0,7.200000e-07,2.300000e-06
697,2026-03-15 14:06:13,2026-04-19T13:39:19.243669Z,z-ai/glm-5-turbo,Z.ai: GLM 5 Turbo,202752.0,1.200000e-06,4.000000e-06
698,2026-04-07 16:07:05,2026-04-19T13:39:19.243669Z,z-ai/glm-5.1,Z.ai: GLM 5.1,202752.0,9.500000e-07,3.150000e-06


In [81]:
llm_models_pricing[llm_models_pricing.model_id.str.startswith("openai")]['model_id'].value_counts()

model_id
openai/gpt-3.5-turbo         2
openai/gpt-audio-mini        2
openai/gpt-5.1-codex-max     2
openai/gpt-5.1-codex-mini    2
openai/gpt-5.2               2
                            ..
openai/gpt-5-image-mini      2
openai/gpt-5-mini            2
openai/gpt-5-nano            2
openai/o4-mini-high          2
openai/gpt-4o:extended       1
Name: count, Length: 62, dtype: Int64

In [90]:
llm_models_pricing[llm_models_pricing.model_id.str.startswith("openai")].sort_values("created_at", ascending=False).head(20)

,created_at,snapshot_ts,model_id,model_name,context_length,pricing_prompt,pricing_completion
578,2026-03-17 11:49:47,2026-04-19T13:39:19.243669Z,openai/gpt-5.4-nano,OpenAI: GPT-5.4 Nano,400000.0,2.000000e-07,0.000001
222,2026-03-17 11:49:47,2026-04-15T22:53:52.430405Z,openai/gpt-5.4-nano,OpenAI: GPT-5.4 Nano,400000.0,2.000000e-07,0.000001
221,2026-03-17 11:49:38,2026-04-15T22:53:52.430405Z,openai/gpt-5.4-mini,OpenAI: GPT-5.4 Mini,400000.0,7.500000e-07,0.000005
577,2026-03-17 11:49:38,2026-04-19T13:39:19.243669Z,openai/gpt-5.4-mini,OpenAI: GPT-5.4 Mini,400000.0,7.500000e-07,0.000005
579,2026-03-05 18:12:46,2026-04-19T13:39:19.243669Z,openai/gpt-5.4-pro,OpenAI: GPT-5.4 Pro,1050000.0,3.000000e-05,0.000180
223,2026-03-05 18:12:46,2026-04-15T22:53:52.430405Z,openai/gpt-5.4-pro,OpenAI: GPT-5.4 Pro,1050000.0,3.000000e-05,0.000180
220,2026-03-05 18:12:32,2026-04-15T22:53:52.430405Z,openai/gpt-5.4,OpenAI: GPT-5.4,1050000.0,2.500000e-06,0.000015
576,2026-03-05 18:12:32,2026-04-19T13:39:19.243669Z,openai/gpt-5.4,OpenAI: GPT-5.4,1050000.0,2.500000e-06,0.000015
574,2026-03-03 18:54:21,2026-04-19T13:39:19.243669Z,openai/gpt-5.3-chat,OpenAI: GPT-5.3 Chat,128000.0,1.750000e-06,0.000014
218,2026-03-03 18:54:21,2026-04-15T22:53:52.430405Z,openai/gpt-5.3-chat,OpenAI: GPT-5.3 Chat,128000.0,1.750000e-06,0.000014


In [93]:
tokens_daily[tokens_daily.provider_slug == "openai"].tail(20)

,usage_date,provider_slug,provider_name,model_permaslug,total_tokens,prompt_tokens,completion_tokens,estimated_revenue,pricing_snapshot_ts,pricing_prompt,pricing_completion,pricing_join_status
2486,2026-04-17,openai,OpenAI,openai/text-embedding-3-small,1.098345e+10,0.0,0.0,214.616526,2026-01-01T00:00:00Z,2.000000e-08,0.000000e+00,matched_model_median
2505,2026-04-18,openai,OpenAI,openai/gpt-4.1-mini-2025-04-14,1.106228e+10,0.0,0.0,4730.230927,2026-04-19T13:39:19Z,4.000000e-07,1.600000e-06,matched_model_median
2506,2026-04-18,openai,OpenAI,openai/gpt-4o-mini,2.849336e+10,0.0,0.0,4568.910155,2026-04-19T13:39:19Z,1.500000e-07,6.000000e-07,matched_model_median
2507,2026-04-18,openai,OpenAI,openai/gpt-5-chat-2025-08-07,9.077788e+09,0.0,0.0,13174.140135,2026-04-19T13:39:19Z,1.250000e-06,1.000000e-05,matched_model_median
2508,2026-04-18,openai,OpenAI,openai/gpt-5-mini-2025-08-07,1.308753e+10,0.0,0.0,3798.654839,2026-04-19T13:39:19Z,2.500000e-07,2.000000e-06,matched_model_median
2509,2026-04-18,openai,OpenAI,openai/gpt-5.3-codex-20260224,1.587361e+10,0.0,0.0,32251.212496,2026-04-19T13:39:19Z,1.750000e-06,1.400000e-05,matched_model_median
2510,2026-04-18,openai,OpenAI,openai/gpt-5.4-20260305,1.081251e+11,0.0,0.0,301398.725747,2026-04-19T13:39:19Z,2.500000e-06,1.500000e-05,matched_model_median
2511,2026-04-18,openai,OpenAI,openai/gpt-5.4-mini-20260317,1.082698e+10,0.0,0.0,9054.063872,2026-04-19T13:39:19Z,7.500000e-07,4.500000e-06,matched_model_median
2512,2026-04-18,openai,OpenAI,openai/gpt-oss-120b,5.068635e+10,0.0,0.0,2152.801237,2026-04-19T13:39:19Z,3.900000e-08,1.900000e-07,matched_model_median
2513,2026-04-18,openai,OpenAI,openai/gpt-oss-20b,1.178675e+09,0.0,0.0,38.342282,2026-04-19T13:39:19Z,3.000000e-08,1.400000e-07,matched_model_median
